# 课程 01 - AI 代理简介

欢迎来到 **AI 新手代理** 课程的第一课！

**AI 代理** 是一个使用大型语言模型（LLM）作为推理引擎的程序，并且能够在现实世界中采取<em>行动</em> —— 调用 API、查询数据库或运行代码 —— 以代表用户完成目标。

在本笔记本中，您将构建第一个代理：一个推荐度假目的地的 <strong>旅行代理</strong>。在此过程中，您将学习如何：

1. 使用 **Microsoft Agent Framework** 连接到 Microsoft Foundry Agent 服务。
2. 给代理一个 <strong>工具</strong> —— 一个它可以调用的普通 Python 函数。
3. 运行代理并检查其响应。
4. 逐个令牌流式传输代理的响应。


## 设置

在运行此笔记本之前，请确保您已完成以下操作：

1. **拥有一个 Microsoft Foundry 项目** 并已部署聊天模型（例如 `gpt-5-mini`）。
2. **已使用 Azure CLI 登录** — 在终端运行 `az login`。
3. **设置必需的环境变量：**
   - `LLM_BASE_URL` — 您的 Microsoft Foundry 项目端点。
   - `LLM_MODEL` — 您已部署模型的名称。

下面的单元格将安装您需要的 Python 包。


In [ ]:
%pip install agent-framework -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.openai import OpenAIChatCompletionClient
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("LLM_BASE_URL")
model = os.getenv("LLM_MODEL")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set LLM_BASE_URL and LLM_MODEL in your .env file."
    )

provider = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

## 创建你的第一个智能体

一个智能体需要两样东西：

- <strong>指令</strong>，告诉它<em>它是谁</em>以及<em>如何表现</em>（系统提示）。
- <strong>工具</strong> —— 用 `@tool` 装饰的 Python 函数，智能体可以调用它们来获取信息或执行操作。

下面我们定义了一个简单的工具，它返回一个受欢迎的度假目的地列表。当用户询问旅行推荐时，智能体将使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "你是一个有用的旅行代理。根据用户的偏好帮他们找到完美的度假目的地。"
        "请使用 get_destinations 工具查看可用目的地列表，并始终用中文回复。"
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "我想去一个温暖的海滩度假地，你推荐哪里？"
)
print(response)

## 流式响应

为了获得更互动的体验，您可以<strong>流式</strong>获取代理的响应。代理会随着文本生成逐块产出，而不是等待完整回复。这在聊天界面中特别有用，因为您希望实时展示输出内容。


In [16]:
async for chunk in agent.run(
    "请用中文介绍一下东京作为旅行目的地", stream=True
):
    print(chunk, end="", flush=True)

东京是日本的首都，也是全球最受欢迎的旅游目的地之一。以下是从多个方面为您介绍的东京旅行亮点：

## 🏯 文化与历史
- **传统与现代的完美融合**：在东京，您可以在同一天参观古老的浅草寺（拥有近1400年历史）和 futuristic 的台场或涩谷天空观景台。
- **皇家风范**：皇居东御苑对外开放，是了解日本皇室历史和欣赏传统日式园林的好去处。
- **传统文化体验**：可以参与茶道体验、歌舞伎表演（在银座的歌舞伎座）或穿着和服在街头漫步。

## 🍣 美食天堂
- **米其林餐厅密度全球领先**：从高级寿司店（如筑地/丰洲市场周边）到平价美味的拉面店、居酒屋，选择极其丰富。
- **街头美食**：章鱼烧、可丽饼、炸鸡（唐扬）等小吃在秋叶原、原宿等地随处可见。
- **主题餐厅**：机器人餐厅、各种动漫/角色主题咖啡馆也是东京独有的体验。

## 🌆 必游景点
- **涩谷十字路口**：世界上最繁忙的十字路口，周边有忠犬八公像和众多商场。
- **新宿**：不夜城区，有热闹的歌舞伎町和宁静的新宿御苑。
- **秋叶原**：动漫、游戏和电子产品爱好者的圣地。
- **明治神宫**：位于代代木公园深处，是感受日本神道文化的宁静绿洲。
- **东京塔 & 东京晴空塔**：两个标志性地标，登塔可俯瞰全城。

## 🛍️ 购物与潮流
- **银座**：高端奢侈品牌和百年老店的聚集地。
- **原宿 & 表参道**：日本潮流文化发源地，适合购买潮牌和设计师品牌。
- **池袋 & 新宿**：大型电器卖场（Bic Camera、Yodobashi）和药妆店集中。

## 🚇 交通与便利
- **轨道交通极其发达**：JR山手线、地铁和私铁组成的网络覆盖全城，购买Suica/Pasmo卡或东京地铁券非常划算。
- **羽田机场和成田机场**：两个国际机场连接全球，羽田距市中心更近。
- **安全与整洁**：东京是全球最安全、最干净的大城市之一，非常适合自由行。

## 🌸 最佳旅行季节
- **春季（3-5月）**：樱花季，上野公园、新宿御苑等地赏樱极佳。
- **秋季（9-11月）**：红叶季节，气候宜人。
- **冬季**：可体验日本新年传统，且购物折扣季（1月）很划算。
- **夏季**：有花火大会，但较为炎热潮湿。

如果您对特定类型的旅行（如亲子游、美食之旅、动漫

## 总结

在本课中，您学到了如何：

- <strong>创建一个提供程序</strong>，通过 `FoundryChatClient` 连接到 Microsoft Foundry Agent Service。
- **使用 `@tool` 装饰器定义工具**，以便代理可以调用您的 Python 函数。
- <strong>运行代理</strong>，发送用户消息并打印其响应。
- <strong>流式传输响应</strong>，实现实时输出。

在下一课中，我们将更深入地探讨代理框架，并学习如何赋予代理更强大的工具和多步骤推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
